# Portfolio construction 1 of 3: what to hold

Notebooks: [1 hold](portfolio_construction_1_hold.ipynb) · [2 scale](portfolio_construction_2_scale.ipynb) · [3 trade](portfolio_construction_3_trade.ipynb), matching `src/portfolio_construction/{hold,scale,trade}`. In-sample, 1976-2014, rolling-covariance baseline (36 months, 50% shrinkage); data conditional on the unverified March 1997 endpoint (`docs/DATA_STATUS.md`).

## Goal

Improve the model. Here: does inverse-volatility sizing leave risk concentrated, and would a better relative sizing improve risk-adjusted return? Judge every change at matched realised volatility, with Sharpe, turnover, cost, drawdown, class risk share and effective bets.

## Current pipeline

**What we are doing.** Choose which markets to hold, long or short, and how large each position is relative to the others. No overall scale yet.

**Inputs**
- Trend signal per market: sign of the lagged 12-month return sum (FX blends 3 and 12 months).
- Forecast annual volatility per market, from the risk model.
- Config: instrument scale 0.40, volatility floor 2%, minimum 10 eligible markets, excluded YM and EC.

**How we process them** (`hold/sizing.py`)
1. **Eligible** if the signal is known, volatility is positive and the market is not excluded. Fewer than 10 eligible markets: the book is flat.
2. **Size** each eligible market as `0.40 / (volatility × number eligible)`, with volatility floored at 2%.
3. **Sign** the size by the trend signal: long, short or flat.
4. *(Off by default)* **Class balance** rescales each asset class to an equal share of the risk ceiling (`hold/class_balance.py`).

*Example:* a market with 10% volatility, 37 eligible markets and a positive signal gets `0.40 / (0.10 × 37)` = 10.8% of capital, long. A market with 20% volatility gets half that.

**Output.** One signed weight per market, passed to the scale stage.

## Implications

- Sizing uses only each market's own volatility. The covariance enters later, as one scalar. The book is inverse-volatility with a global scale, not an optimisation.
- Correlated clusters get weight by headcount (22 commodity markets of about 37 active).
- No return forecast enters sizing, so no Markowitz error maximisation. Covariance instability only matters for edits that invert or use correlations.

## Results and interpretation

Euler risk contribution `rc_i = w_i (Qw)_i / w'Qw`. Effective bets = `1 / sum(rc_i²)`. Condition number of the active covariance.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]


from risk_model_study import load_study, rolling_model
from src.backtest.qps_baseline import forecasts_from, qps_components

pd.options.display.float_format = "{:.3f}".format
study = load_study()
settings, base, returns, source, known, meta = study.inputs
model = rolling_model(study)
parts = qps_components(base, settings, meta, model)
targets, covariance, _ = forecasts_from(parts, returns, meta, base)
labels = meta.AssetClass.reindex(returns.columns)
print(f"{returns.shape[1]} markets; trend target months: {len(targets['trend'])}")

58 markets; trend target months: 468


In [3]:
trend = targets["trend"]
rows = []
for month in trend.index:
    w = trend.loc[month].to_numpy()
    active = w != 0
    if active.sum() < base["minimum_instruments"]:
        continue
    Q = covariance[month]
    contribution = w * (Q @ w) / (w @ Q @ w)
    by_class = pd.Series(contribution, index=trend.columns).groupby(labels).sum()
    eig = np.linalg.eigvalsh(Q[np.ix_(active, active)])
    rows.append(
        {
            "month": month,
            "markets": int(active.sum()),
            "effective_bets": 1 / np.sum(contribution[active] ** 2),
            "condition_number": eig.max() / eig.min(),
            **by_class.add_prefix("risk_share_").to_dict(),
        }
    )
shape = pd.DataFrame(rows).set_index("month")
shape["distinct_cov_entries"] = shape.markets * (shape.markets + 1) / 2
shape["effective_bets_ratio"] = shape.effective_bets / shape.markets
shape.describe().T[["mean", "min", "50%", "max"]]

,mean,min,50%,max
markets,36.994,11.000,39.000,56.000
effective_bets,24.219,7.523,24.832,47.415
condition_number,1198.031,11.112,1021.418,4653.819
risk_share_COMM,0.589,0.072,0.560,1.000
risk_share_EQ,0.164,-0.002,0.085,0.628
risk_share_FI,0.139,-0.059,0.098,0.543
risk_share_FX,0.108,-0.024,0.098,0.317
distinct_cov_entries,797.146,66.000,780.000,1596.000
effective_bets_ratio,0.680,0.295,0.694,0.935


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
share = shape.filter(like="risk_share_").rename(columns=lambda c: c.replace("risk_share_", ""))
share.index = share.index.to_timestamp() if hasattr(share.index, "to_timestamp") else share.index
share.plot(ax=axes[0], title="Share of forecast risk by asset class")
axes[1].plot(share.index, shape.markets, label="active markets")
axes[1].plot(share.index, shape.effective_bets, label="effective bets (1/HHI of risk)")
axes[1].set_title("Diversification actually obtained"); axes[1].legend()
axes[2].semilogy(share.index, shape.condition_number)
axes[2].set_title("Condition number of active covariance")
plt.tight_layout(); plt.show()

/var/folders/2s/4bwc4_c534sg_8zdq7wkq00c0000gn/T/ipykernel_94651/118578350.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


- **Commodities carry about 59% of forecast risk**, with 22 of about 37 active markets.
- **Effective bets average about 24** against about 37 markets (68%), and fall to about 7.5 at the minimum.
- **Condition number averages about 1,200** (peak about 4,700): about 800 covariance entries from 36 months. Harmless to sizing today. It matters for HRP or mean-variance.

## AFML improvements

Read: Ch. 16 (16.2-16.6), Ch. 14 (14.5.1, 14.7.3).

| Idea | What the book says | Use here |
|---|---|---|
| Markowitz's curse (16.3) | Correlated assets make the inverted covariance unstable; a non-singular `N×N` needs at least `N(N+1)/2` IID observations. | Condition number about 1,200 on 36 months: avoid mean-variance here. |
| Hierarchical Risk Parity (16.4) | Cluster by `sqrt((1-ρ)/2)`, quasi-diagonalise, split weight top-down by cluster variance. No inversion. | A data-driven version of class balance for steps 3-4. |
| Out-of-sample test (16.5-16.6) | Monte Carlo, 10 assets, 10,000 runs: out-of-sample variance 0.1157 (min-variance CLA), 0.0928 (inverse-variance), 0.0671 (HRP). | Judge edits out of sample. Simulated Gaussian data: a design hint only. |
| Bet concentration (14.5.1) | HHI-style concentration of bets. | Applied here to risk contributions. |
| Deflated Sharpe (14.7.3) | Judge a Sharpe against the maximum expected from the number of trials. | Every variant is a trial on an inspected window. |

## Suggested changes and experiments

| # | Change | Experiment | Effort |
|---|---|---|---|
| P0 | Add diagnostics to every run | Record class risk share, effective bets and trial count for the baseline as the reference. No change to `src/`. | Low |
| P1 | Class risk budgets | Run the existing `class_balanced` model. Check it can raise a quiet class above its raw size; if so, cap it to scale down only. Then try a per-class cap (e.g. 40% of risk). Net of costs. | Low |
| P2 | HRP weights replace inverse-vol | Correlate returns × lagged signal sign, cluster, bisect, use as relative sizes. Keep scale and trade stages unchanged. Start within class. Compare with P1. | Medium |
| P3 | Out-of-sample construction test | Block-bootstrap monthly returns; compare inverse-vol, class-balanced and HRP on out-of-sample variance, turnover and concentration across draws. | Medium |
| P5 | Choose by CPCV, then deflate | Pick the class cap or HRP variant with `src/backtest/cross_validation.py`; report the deflated Sharpe with the trial count. | Medium |

**Not recommended:** mean-variance or minimum-variance on this covariance (condition number, 36 months, no return forecast to optimise).

**Order:** P0, then P1, then P2 judged against P1. If class budgets give most of the gain, HRP adds complexity for little. Size discretisation (P4) is in notebook 3.

**Open questions:** (1) Hold *realised* or *forecast* volatility fixed when comparing? I assumed realised. (2) For P2, cluster on signed returns or raw returns? (3) Is a class cap a hard limit or a target?